# 04 - Baseline comparison

Runs locally, no GPU, seconds.

**The task.** Predict, for a held-out entity, whether the model will produce a wrong answer.
Labels come from **free generation**; features come from the **constrained probe** and from
**corpus statistics**. Two different measurement channels, so no predictor can see its own label.

The version in the original plan defined the label by the same rule as the predictor, which
scores 100% by construction and measures nothing. This is the de-circularised version.

Thresholds are tuned on the validation split and reported on test; splits are grouped by
**subject entity**, so no entity appears in more than one split.

## 1. Setup

In [ ]:

import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# ---- paper-figure defaults (ACL: \columnwidth ~3.17in, \textwidth ~6.3in) ----
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "grid.linewidth": 0.4, "lines.linewidth": 1.4,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
COL, FULL = 3.17, 6.3
SURFACE = "#fcfcfb"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#8a8984"
# categorical slots 1-3 (validated all-pairs, light)
CAT = ["#2a78d6", "#eb6834", "#1baf7a"]
# 5-step ordinal blue ramp (validated: monotone L, adjacent dL >= 0.06)
ORD5 = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
# diverging blue<->red with neutral gray midpoint
DIV = LinearSegmentedColormap.from_list("bl_rd",
        ["#0d366b", "#2a78d6", "#86b6ef", "#f0efec", "#f0a19f", "#d03b3b", "#7d1f1f"])
SEQ = LinearSegmentedColormap.from_list("blues", ["#eef4fd", "#86b6ef", "#2a78d6", "#0d366b"])

BUCKETS = ["0","1","2","3-4","5-7","8-13","14-25","26-60","61-200","200+"]
BSHOW   = ["0","2","5-7","26-60","200+"]          # 5 representative buckets for line charts
MAIN, FINAL = "LMEnt-1B-6E", 658032

def style(ax, grid="y"):
    ax.set_facecolor(SURFACE)
    if grid: ax.grid(axis=grid, color="#e3e2de", zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, length=2, width=0.6)
    for s in ax.spines.values(): s.set_color("#c9c8c3")
    return ax

def save(fig, name, outdir):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(outdir, f"{name}.{ext}"), format=ext)
    plt.close(fig)
    print("  figure:", name + ".pdf")

def load_results(root="runs"):
    fs = sorted(glob.glob(os.path.join(root, "*", "results", "res__*.parquet")))
    assert fs, f"no result parquets under {root}/*/results/"
    df = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
    num = [c for c in df.columns if df[c].dtype.kind == "f"]
    assert not df[num].isna().any().any(), "NaN in results"
    assert df.duplicated(["model","templates","step","fact_id"]).sum() == 0, "duplicate rows"
    n = df.groupby(["model","templates","step"]).fact_id.nunique().unique()
    assert len(n) == 1, f"inconsistent fact counts per checkpoint: {n}"
    df["bucket"] = pd.Categorical(df.bucket, BUCKETS, ordered=True)
    df["gap_f"] = df.conf_norm - df.acc_norm          # fact-level gap
    return df, fs

def gap_table(d, index="bucket", col="step"):
    g = (d.groupby([index, col], observed=True).conf_norm.mean()
         - d.groupby([index, col], observed=True).acc_norm.mean())
    out = g.unstack(col)
    return out.reindex(BUCKETS) if index == "bucket" else out

def boot_ci(x, n=2000, seed=0, stat=np.mean):
    r = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    if len(x) == 0: return (np.nan, np.nan)
    bs = stat(x[r.integers(0, len(x), (n, len(x)))], axis=1)
    return tuple(np.percentile(bs, [2.5, 97.5]))

def ece(conf, correct, bins=10):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    edges = np.linspace(0, 1, bins+1); tot = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i+1]) if i else (conf <= edges[1])
        if m.sum(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return tot


In [ ]:
import os, json
FIGDIR, TABDIR = "analysis/figures", "analysis/tables"
os.makedirs(FIGDIR, exist_ok=True); os.makedirs(TABDIR, exist_ok=True)
T = {}
df, files = load_results("runs")
print(f"{len(files)} files, {len(df):,} rows")

## 2. Two questions

The class base rate is ~86%, so accuracy and F1 are nearly uninformative here - a predictor that
always says "wrong" gets F1 0.927. **ROC-AUC and balanced accuracy are the meaningful columns.**

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             balanced_accuracy_score, precision_recall_fscore_support)

# ---------------------------------------------------------------------------
# LABELS come from FREE GENERATION; FEATURES from the constrained probe and the
# corpus. Different measurement channels, so no predictor can see its own label.
# (Defining the label by the same rule as the predictor - as the original plan
#  did - scores 100% by construction and measures nothing.)
# ---------------------------------------------------------------------------
d = df[(df.model==MAIN)&(df.templates=="main")].copy()
d["logexp"] = np.log1p(d.n_shared)
tr_all = d[d.split=="train"]

# Threshold at which the model's free generation is right ~50% of the time.
# Derived from the train split only; this is the model's own "I am sure" level.
_q = np.linspace(0.5, 0.99, 50)
_c = [(tr_all.gen_conf.quantile(q), tr_all[tr_all.gen_conf > tr_all.gen_conf.quantile(q)].gen_correct.mean())
      for q in _q]
TAU_CAL = min(_c, key=lambda t: abs(t[1]-0.5))[0]
print(f"calibrated confidence threshold: gen_conf > {TAU_CAL:.3f} "
      f"(where free generation is right ~50% of the time)")

d["err"]      = (d.gen_correct == 0).astype(int)              # task 1: will it be wrong?
d["conf_err"] = ((d.gen_correct == 0) & (d.gen_conf > TAU_CAL)).astype(int)   # task 2

def metrics(y, score, thr):
    yh = (score > thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, yh, average="binary", zero_division=0)
    return {"acc": accuracy_score(y, yh), "bal_acc": balanced_accuracy_score(y, yh),
            "prec": p, "rec": r, "f1": f,
            "auc": roc_auc_score(y, score) if len(set(y)) > 1 else np.nan,
            "ap": average_precision_score(y, score) if len(set(y)) > 1 else np.nan}

def tune(y, score):
    grid = np.quantile(score, np.linspace(0.02, 0.98, 49))
    return max(grid, key=lambda t: precision_recall_fscore_support(
        y, (score > t).astype(int), average="binary", zero_division=0)[2])

def run_task(label, step=None):
    step = step or FINAL
    s = d[d.step == step]
    tr, va, te = (s[s.split==k] for k in ("train","val","test"))
    base = te[label].mean()
    out = {}

    maj = int(tr[label].mean() > 0.5)
    yh  = np.full(len(te), maj)
    p, r, f, _ = precision_recall_fscore_support(te[label], yh, average="binary", zero_division=0)
    out["majority class"] = {"acc": accuracy_score(te[label], yh),
                             "bal_acc": balanced_accuracy_score(te[label], yh),
                             "prec": p, "rec": r, "f1": f, "auc": 0.5, "ap": base}

    feats = {"frequency only": (-tr.logexp, -va.logexp, -te.logexp),
             "confidence only": (-tr.conf_norm, -va.conf_norm, -te.conf_norm)}
    for name, (a, b, c) in feats.items():
        out[name] = metrics(te[label], c, tune(va[label], b))

    lr = LogisticRegression(max_iter=1000).fit(tr[["conf_norm","logexp"]], tr[label])
    pv, pt = (lr.predict_proba(x[["conf_norm","logexp"]])[:,1] for x in (va, te))
    out["ours (both)"] = metrics(te[label], pt, tune(va[label], pv))
    out["ours (both)"]["coef"] = [float(x) for x in lr.coef_[0]]

    tab = pd.DataFrame(out).T[["acc","bal_acc","prec","rec","f1","auc","ap"]].astype(float)
    print(f"\n--- {label} at step {step:,} | test n={len(te)}, positive rate {base:.1%} ---")
    print(tab.round(3).to_string())
    return tab

print("\n" + "="*70)
print("TASK 1: will the model get this fact wrong in free generation?")
print("="*70)
TAB_ERR = run_task("err")

# Task 2 is NOT posed as a prediction task: at a calibrated threshold only ~1% of
# test facts qualify, so any metric on it would be noise. It is reported descriptively,
# at several thresholds, because the SHAPE is the finding.
print("\n" + "="*70)
print("CONFIDENT ERRORS: descriptive, at three thresholds")
print("="*70)
f = d[d.step==FINAL]
rates = {}
for q in (0.50, 0.75, 0.90):
    t = tr_all.gen_conf.quantile(q)
    lab = ((f.gen_correct==0) & (f.gen_conf > t)).astype(int)
    rates[f"q{int(q*100)}"] = f.assign(z=lab).groupby("bucket", observed=True).z.mean().reindex(BUCKETS)
RATES = pd.DataFrame(rates)
RATES["err"] = f.groupby("bucket", observed=True).gen_correct.apply(lambda x: 1-x.mean()).reindex(BUCKETS)
print(RATES.round(3).to_string())
print("\nErrors fall monotonically with exposure. Confident errors do NOT - they peak at")
print("intermediate exposure at every threshold. Producing a confident wrong answer needs")
print("enough knowledge to form a specific one; the rarest entities yield hesitant errors.")
by_b = RATES

T["baselines_err"] = TAB_ERR; T["rate_by_bucket"] = RATES


## 3. Figure

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.1))

ax = style(axes[0])
tab = TAB_ERR.sort_values("auc")
y = np.arange(len(tab))
cols = [MUTED if i=="majority class" else (CAT[0] if i.startswith("ours") else CAT[2]) for i in tab.index]
ax.barh(y, tab.auc.values, color=cols, height=0.6, zorder=3, edgecolor=SURFACE, linewidth=0.8)
for yi, v in zip(y, tab.auc.values):
    ax.annotate(f"{v:.3f}", (v, yi), xytext=(3,0), textcoords="offset points",
                va="center", fontsize=6.5, color=INK2)
ax.axvline(0.5, color=INK, lw=0.7, ls="--")
ax.set_yticks(y); ax.set_yticklabels(tab.index)
ax.set_xlabel("ROC-AUC, predicting a wrong answer"); ax.set_xlim(0.4, 1.0)
ax.set_title("errors are predictable", fontsize=7.5, color=INK2)

ax = style(axes[1], grid="both")
x = np.arange(len(BUCKETS))
ax.plot(x, RATES.err.values, color=CAT[1], marker="s", ms=2.8, label="wrong")
ax.plot(x, RATES.q75.values, color=CAT[0], marker="o", ms=2.8, label="wrong AND confident")
ax.set_xticks(x); ax.set_xticklabels(BUCKETS, rotation=45, ha="right")
ax.set_xlabel("co-occurrences"); ax.set_ylabel("rate in free generation")
ax.legend(frameon=False, loc="center left"); ax.set_ylim(0, 1.02)
ax.set_title("but confident errors peak in the middle", fontsize=7.5, color=INK2)
save(fig, "fig9_baselines", FIGDIR)


## 4. Export

In [ ]:
for k, v in T.items(): v.to_csv(os.path.join(TABDIR, f"table_{k}.csv"))
print("exported:", [f"table_{k}.csv" for k in T])